# 1. Clean Data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
%matplotlib inline 
import matplotlib.pyplot as plt
import plotly.express as px
import sklearn

In [2]:
cd C:\Users\Lenovo\OneDrive\Desktop

C:\Users\Lenovo\OneDrive\Desktop


In [26]:
data = pd.read_csv("311data.csv")
data.shape

(800000, 29)

In [27]:
data.columns

Index(['_id', 'group_id', 'num_requests', 'parent_closed', 'status_name',
       'status_code', 'dept', 'request_type_name', 'request_type_id',
       'create_date_et', 'create_date_utc', 'last_action_et',
       'last_action_utc', 'closed_date_et', 'closed_date_utc', 'origin',
       'street', 'cross_street', 'street_id', 'cross_street_id', 'city',
       'neighborhood', 'census_tract', 'council_district', 'ward',
       'police_zone', 'latitude', 'longitude', 'geo_accuracy'],
      dtype='object')

In [28]:
data.isnull().sum()

_id                       0
group_id                  0
num_requests              0
parent_closed             0
status_name               0
status_code               0
dept                   4103
request_type_name         0
request_type_id           0
create_date_et            0
create_date_utc           0
last_action_et            0
last_action_utc           0
closed_date_et        89049
closed_date_utc       89049
origin                    0
street               308423
cross_street         698878
street_id            301948
cross_street_id      301948
city                      0
neighborhood          36109
census_tract         218417
council_district      34677
ward                  35943
police_zone           36202
latitude              30175
longitude             30175
geo_accuracy              0
dtype: int64

## (1) Drop columns

In [29]:
data = data.drop(columns=['create_date_utc', 
                          'last_action_utc', 
                          'closed_date_utc',
                          'cross_street', 
                          'street', 
                          'street_id', 
                          'cross_street_id',
                         'census_tract'])

In [30]:
data.isnull().sum()

_id                      0
group_id                 0
num_requests             0
parent_closed            0
status_name              0
status_code              0
dept                  4103
request_type_name        0
request_type_id          0
create_date_et           0
last_action_et           0
closed_date_et       89049
origin                   0
city                     0
neighborhood         36109
council_district     34677
ward                 35943
police_zone          36202
latitude             30175
longitude            30175
geo_accuracy             0
dtype: int64

## (2) convert time

In [31]:
data['create_date_et'] = pd.to_datetime(data['create_date_et'])
data['closed_date_et'] = pd.to_datetime(data['closed_date_et'])

In [32]:
data = data.sort_values(by='create_date_et', ascending=True)

In [33]:
data['create_date_et']

151382   2015-04-20 07:37:00
173037   2015-04-20 07:39:00
196183   2015-04-20 07:40:00
111096   2015-04-20 07:41:00
413448   2015-04-20 07:46:00
                 ...        
799997   2024-12-18 09:28:00
799994   2024-12-18 09:32:00
799993   2024-12-18 09:39:00
799990   2024-12-18 09:40:00
799999   2024-12-18 09:41:00
Name: create_date_et, Length: 800000, dtype: datetime64[ns]

## (3) Fill in neighbourhood

In [34]:
# Reverse geocoding function
def get_neighborhood(lat, lon):
    try:
        location = geolocator.reverse(f"{lat}, {lon}", exactly_one=True)
        return location.raw['address'].get('neighbourhood', 'Unknown')
    except:
        return 'Unknown'

#Execute only on data that lacks neighborhood and has longitude and latitude
mask = data['neighborhood'].isna() & data['latitude'].notna() & data['longitude'].notna()
data.loc[mask, 'neighborhood'] = data[mask].apply(
    lambda row: get_neighborhood(row['latitude'], row['longitude']), axis=1
)

In [35]:
data.isnull().sum()

_id                      0
group_id                 0
num_requests             0
parent_closed            0
status_name              0
status_code              0
dept                  4103
request_type_name        0
request_type_id          0
create_date_et           0
last_action_et           0
closed_date_et       89049
origin                   0
city                     0
neighborhood         30155
council_district     34677
ward                 35943
police_zone          36202
latitude             30175
longitude            30175
geo_accuracy             0
dtype: int64

## (4) fill in concil_district

In [36]:
data['council_district'].unique()

array([ 7., nan,  3.,  8.,  9.,  2.,  1.,  5.,  6.,  4.])

In [37]:
mapping = {
    "Knoxville": 3,
    "Regent Square": 5,
    "Carrick": 4,
    "Fairywood": 2,
    "Oakwood": 2,
    "Lincoln Place": 5,
    "Brookline": 4,
    "East Hills": 9,
    "Westwood": 2,
    "Point Breeze": 8,
    "Homewood South": 9,
    "Overbrook": 4,
    "Spring Garden": 1,
    "Beechview": 4,
    "Perry North": 1,
    "Point Breeze North": 8,
    "Squirrel Hill North": 8,
    "Arlington": 3,
    "East Carnegie": 2,
    "East Liberty": 9,
    "North Oakland": 8,
    "Strip District": 7,
    "Summer Hill": 1}

In [38]:
missing_mask = data['council_district'].isna()

data.loc[missing_mask, 'council_district'] = data.loc[missing_mask, 'neighborhood'].map(mapping)

In [39]:
data.isnull().sum()

_id                      0
group_id                 0
num_requests             0
parent_closed            0
status_name              0
status_code              0
dept                  4103
request_type_name        0
request_type_id          0
create_date_et           0
last_action_et           0
closed_date_et       89049
origin                   0
city                     0
neighborhood         30155
council_district     34532
ward                 35943
police_zone          36202
latitude             30175
longitude            30175
geo_accuracy             0
dtype: int64

## (5) parent_closed

In [40]:
data['parent_closed'] = data['parent_closed'].replace({'t': 1, 'f': 0})

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6640\3555678057.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['parent_closed'] = data['parent_closed'].replace({'t': 1, 'f': 0})


In [41]:
data['parent_closed']

151382    1
173037    1
196183    1
111096    1
413448    1
         ..
799997    1
799994    1
799993    1
799990    0
799999    1
Name: parent_closed, Length: 800000, dtype: int64

## (6) Origin

In [42]:
# One-Hot
origin_dummies = pd.get_dummies(data['origin'], prefix='origin', dtype=int)


data = pd.concat([data, origin_dummies], axis=1)
data[['origin', 'origin_Call Center', 'origin_Website']].head()

,origin,origin_Call Center,origin_Website
151382,Call Center,1,0
173037,Call Center,1,0
196183,Call Center,1,0
111096,Call Center,1,0
413448,Call Center,1,0


In [43]:
data.columns

Index(['_id', 'group_id', 'num_requests', 'parent_closed', 'status_name',
       'status_code', 'dept', 'request_type_name', 'request_type_id',
       'create_date_et', 'last_action_et', 'closed_date_et', 'origin', 'city',
       'neighborhood', 'council_district', 'ward', 'police_zone', 'latitude',
       'longitude', 'geo_accuracy', 'origin_Call Center',
       'origin_Control Panel', 'origin_Email', 'origin_QAlert Mobile iOS',
       'origin_Report2Gov Android', 'origin_Report2Gov Website',
       'origin_Report2Gov iOS', 'origin_Text Message', 'origin_Twitter',
       'origin_Website'],
      dtype='object')

## (7) Request_id

In [44]:
name_id_mapping = data.groupby('request_type_name')['request_type_id'].nunique()
name_id_mapping[name_id_mapping > 1]

request_type_name
Crosswalk, New                         2
Dumpster                               2
Earned Income Tax                      2
Electronic/Hazardous Waste Disposal    2
Illegal Dumping                        2
Illegal Parking                        2
Leak                                   2
Litter Can                             2
Litter Can, Public                     2
Potholes                               2
Real Estate Tax                        2
Refuse Violations                      2
Request New Sign                       2
Thank You                              2
Water/Drinking Fountains               2
Name: request_type_id, dtype: int64

396224（1个）,464(7个）
33748 （8个）, 16214 （10个）
370（215个）, 14565 (7个）
468456（17个），362488（2个）
828（187个），160776（5个）
417（207个），160754（2个）
394（22个）,525（16个）
429（7个）,478（3个）
833（41个）,478（3个）
484（783个），160741（19个）
14568（221个），373（21个）
512（201个），160755（4个）
270186（28个），491（28个）
12074（261个）,255453（49个）
40178（15个），827（14个）

In [45]:
correct_mapping = {
    'Crosswalk, New': 464,
    'Dumpster': 16214,
    'Earned Income Tax': 370,
    'Electronic/Hazardous Waste Disposal': 468456,
    'Illegal Dumping': 828,
    'Illegal Parking': 417,
    'Leak': 394,
    'Litter Can': 429,
    'Litter Can, Public': 833,
    'Potholes': 484,
    'Real Estate Tax': 14568,
    'Refuse Violations': 512,
    'Request New Sign': 270186,
    'Thank You': 12074,
    'Water/Drinking Fountains': 40178
}

In [46]:
def correct_request_type_id(row):
    correct_id = correct_mapping.get(row['request_type_name'])
    if correct_id is not None and row['request_type_id'] != correct_id:
        return correct_id
    else:
        return row['request_type_id']
        
data['request_type_id_corrected'] = data.apply(correct_request_type_id, axis=1)

In [47]:
name_id_mapping = data.groupby('request_type_name')['request_type_id_corrected'].nunique()
name_id_mapping[name_id_mapping != 1]

Series([], Name: request_type_id_corrected, dtype: int64)

## (8) request_type

In [48]:
data['request_type_name'].nunique()

345

In [49]:
data['request_type_name'].unique()

array(['Replace/Repair a Sign', 'Permits, Licenses and Inspections',
       'Litter, Public Property', 'Potholes', 'Illegal Dumping',
       'Leaves/Street Cleaning', 'Retaining Wall',
       'Tree Fallen Across Road', 'Public Right of Way',
       'Crosswalk and Street Markings, Maintenance', 'Request New Sign',
       'Refuse Violations', 'DO NOT USE (Vacant and Open Building)',
       'Missed Refuse Pick Up', 'Pruning (city tree)', 'Field',
       'Dead tree (Public property)', 'Street Cleaning/Sweeping',
       'Root prune', 'Overgrowth', 'Abandoned Vehicle (parked on street)',
       'Tree Fallen Across Sidewalk', 'ADA Ramp, Installation',
       'Curb/Request for Asphalt Windrow', 'Tree Removal',
       'Paving Request', 'Dumpster (on Street)', 'Planting',
       'Utility Cut - PWSA', 'Pedestrian Signal Request', 'Road',
       'Sidewalk/Curb/ADA Ramp Maintenance', 'Curb Cuts',
       'Stump Grind/Removal', 'Utility Pole',
       'Traffic or Pedestrian Signal, Request', 'Traffic 

In [51]:
type_counts = data['request_type_name'].value_counts(normalize=True)

# Top 60 High frequency types
type_counts.head(60)

request_type_name
Weeds/Debris                            0.096412
Potholes                                0.083590
Missed Refuse Pick Up                   0.051264
Snow/Ice removal                        0.039506
Building Maintenance                    0.035799
Refuse Violations                       0.032379
Abandoned Vehicle (parked on street)    0.029345
Illegal Parking                         0.026050
Litter, Public Property                 0.022017
Street Light - Repair                   0.021898
Missed Recycling Pick Up                0.021438
Replace/Repair a Sign                   0.017128
Building Without a Permit               0.014615
Overgrowth                              0.013458
Paving Request                          0.011865
Pruning (city tree)                     0.010988
City Source (CDBG)                      0.010807
Early Set Out                           0.010505
Excessive Noise/Disturbances            0.009936
Dead Animal                             0.009832
St

In [52]:
data = data[~data['request_type_name'].str.contains(r'\(DO NOT USE\)', case=False, na=False)]

In [53]:
data['request_type_name'].nunique()

333

## (9) time

In [55]:
closed_requests = data[data['closed_date_et'].notna()].copy()
open_requests = data[data['closed_date_et'].isna()].copy()

In [56]:
data.shape

(799566, 32)

In [57]:
print(closed_requests.shape,closed_requests.shape[0]/800000)
print(open_requests.shape,open_requests.shape[0]/800000)

(710557, 32) 0.88819625
(89009, 32) 0.11126125


### i. Non-closed request

In [58]:
status_dist = open_requests['status_name'].value_counts(normalize=True).mul(100).round(2)
print(f"status distribution（%）:\n{status_dist.to_string()}")

status distribution（%）:
status_name
in progress    51.40
open           48.08
on hold         0.52


In [59]:
for status in ['open', 'in progress', 'on hold']:
    status_subset = open_requests[open_requests['status_name'] == status]
    top_types = status_subset['request_type_name'].value_counts().head(10)
    print(f"\n{status}: top 5 request types")
    print(top_types.to_string())


open: top 5 request types
request_type_name
Abandoned Vehicle (parked on street)    7866
Illegal Parking                         2477
SPIN (Stand Up) Scooters                2211
Drug Enforcement                        1924
Dashcam                                 1768
City Source (CDBG)                      1390
Speeding                                1151
City Owned Property Maintenance          945
Snow/Ice removal                         919
Health Hazard                            738

in progress: top 5 request types
request_type_name
Weeds/Debris                            11122
Building Maintenance                     6430
Abandoned Vehicle (parked on street)     1928
Building Without a Permit                1871
Illegal Parking                          1609
Vacant Building                          1584
Broken Sidewalk                          1053
Pruning (city tree)                       748
Patrol                                    690
Excessive Noise/Disturbances           

### ii. Closed request

In [60]:
closed_top = closed_requests['request_type_name'].value_counts().head(10)
print(f"closed requests top 10:\n{closed_top.to_string()}")

closed requests top 10:
request_type_name
Potholes                    66601
Weeds/Debris                65685
Missed Refuse Pick Up       40335
Snow/Ice removal            30354
Refuse Violations           25238
Building Maintenance        22108
Street Light - Repair       17263
Missed Recycling Pick Up    16951
Litter, Public Property     16869
Illegal Parking             16718


In [61]:
closed_requests['time_taken'] = (closed_requests['closed_date_et'] - closed_requests['create_date_et']).dt.total_seconds() / 3600
closed_requests['time_taken']

151382      222.800000
173037     5069.000000
196183     8622.066667
111096      554.850000
413448    21049.116667
              ...     
799995       49.983333
799997        0.250000
799994       17.983333
799993       53.166667
799999     2015.050000
Name: time_taken, Length: 710557, dtype: float64

In [62]:
time_stats = closed_requests.groupby('request_type_name')['time_taken'].agg(
    ['count', 'mean', 'median', 'min', 'max', 'std']
).sort_values('mean', ascending=False)

# 显示处理时间最长的类型
print("\n平均处理时间最长的10种请求:")
print(time_stats.head(10).to_string())

# 显示处理时间最短的类型
print("\n平均处理时间最短的10种请求:")
print(time_stats[time_stats['count'] > 50].tail(10).to_string())  # 过滤低频类型

# 异常值检查（超过30天的请求）
long_requests = closed_requests[closed_requests['time_taken'] > 720]  # 30天=720小时
print(f"\n超过30天处理的请求数量: {len(long_requests)}")
print("这些请求的类型分布:")
print(long_requests['request_type_name'].value_counts().head(10).to_string())


平均处理时间最长的10种请求:
                                                 count          mean        median           min           max           std
request_type_name                                                                                                           
Retaining Wall (Public)                            197  24692.589932   7469.116667      0.500000  77955.416667  26245.410870
Bicycle/Pedestrian/Trail - Network Improvements      2  22467.908333  22467.908333  22442.800000  22493.016667     35.508546
Water Runoff in ROW                                150  18896.862556  18671.408333      0.000000  43938.000000  12142.269429
Sidewalk/Curb/ADA Ramp Maintenance                2365  17848.397364  19323.350000      0.000000  84715.950000  13362.061815
Traffic Signals Surtrac                             90  16050.143704  18050.341667      0.050000  29014.466667   8419.948796
County Maintenance                                 275  15544.423273  10782.583333      0.183333  53788.6666

In [63]:
type_counts2 = closed_requests['request_type_name'].value_counts(normalize=True)

# Top 60 High frequency types
type_counts2.head(60)

request_type_name
Potholes                                      0.093731
Weeds/Debris                                  0.092442
Missed Refuse Pick Up                         0.056765
Snow/Ice removal                              0.042719
Refuse Violations                             0.035519
Building Maintenance                          0.031114
Street Light - Repair                         0.024295
Missed Recycling Pick Up                      0.023856
Litter, Public Property                       0.023741
Illegal Parking                               0.023528
Abandoned Vehicle (parked on street)          0.019255
Replace/Repair a Sign                         0.018989
Overgrowth                                    0.014822
Building Without a Permit                     0.013530
Paving Request                                0.013288
Early Set Out                                 0.011536
Pruning (city tree)                           0.011305
Dead Animal                                   0

### Combine to "Others"

In [69]:
# 定义阈值（例如占比<0.1%的类别视为低频）
threshold = 0.002
low_freq_types = type_counts2[type_counts2 < threshold].index.tolist()
low_freq_types

['Check Conditions',
 'Graffiti, Removal',
 'Traffic Improvement',
 'City Owned Property Maintenance',
 'Dead Tree (3TB)',
 'Field',
 'Late Set Out',
 'Loose Dog(s)',
 'Repair City Steps',
 'Manhole Covers, PWSA',
 'Animal Waste',
 'Public Works Department',
 'Traffic Signal Request',
 'Curb Paint, Maintenance',
 'Sewers',
 'Schedule Request',
 'Overgrowth, Parks',
 'Boat/Trailer on Street',
 'Operating Without a License',
 'Lights',
 'Board Up (City-owned property only)',
 'Dumpster',
 'URA property',
 'Bicycle/Pedestrian Concerns',
 'Salt Box',
 'Retaining Wall Maintenance',
 'Manhole Cover',
 'Litter',
 'Park Shelter',
 'Stormwater Runoff',
 'Sidewalk, Lack of Snow/Ice Removal',
 'Water/Drinking Fountains',
 'Permits, Licenses and Inspections',
 'City Cuts Concern',
 '911 Performance',
 'Need Potable Water',
 'Cable Bureau/Programming',
 'Noise',
 'Bike Lane Bollard',
 'Gang Activity',
 'Handicap Parking Signs, Removal',
 'Unpermitted HVAC Work',
 'Street Light - Request',
 'Parks T

In [70]:
# 合并低频类别为"Other"
closed_requests['request_type_grouped'] = closed_requests['request_type_name'].apply(
    lambda x: x if x not in low_freq_types else 'Other'
)


'# 步骤2：合并相似类别
category_merge_map = {
    'Potholes': 'Road_Issues',
    'Pothole Repair': 'Road_Issues',
    'Illegal Dumping': 'Waste_Issues',
    'Garbage Pickup': 'Waste_Issues'
}
df['request_type_grouped'] = df['request_type_grouped'].replace(category_merge_map)



In [71]:
closed_requests['request_type_grouped'].unique()

array(['Replace/Repair a Sign', 'Other', 'Litter, Public Property',
       'Potholes', 'Illegal Dumping', 'Leaves/Street Cleaning',
       'Tree Fallen Across Road', 'Public Right of Way',
       'Crosswalk and Street Markings, Maintenance', 'Request New Sign',
       'Refuse Violations', 'Missed Refuse Pick Up',
       'Pruning (city tree)', 'Dead tree (Public property)',
       'Street Cleaning/Sweeping', 'Root prune', 'Overgrowth',
       'Abandoned Vehicle (parked on street)',
       'Tree Fallen Across Sidewalk', 'Curb/Request for Asphalt Windrow',
       'Tree Removal', 'Paving Request', 'Dumpster (on Street)',
       'Utility Cut - PWSA', 'Road', 'Sidewalk/Curb/ADA Ramp Maintenance',
       'Utility Pole', 'Traffic or Pedestrian Signal, Request',
       'Blocked or Closed Sidewalks', 'Missed Recycling Pick Up',
       'Traffic or Pedestrian Signal, Repair', 'Drainage/Leak',
       'Dead Animal', 'Litter Can, Public', 'Commercial Refuse/Dumpsters',
       'Catch Basin, Clogged', 

In [ ]:
# 步骤3：One-Hot编码
type_dummies = pd.get_dummies(closed_requests['request_type_grouped'], prefix='type', dtype=int)
closed_requests2 = closed_requests
closed_requests2 = closed_requests2.concat([closed_requests, type_dummies], axis=1)

start_date = pd.to_datetime('2020-01-01')
end_date = pd.to_datetime('2024-12-31')

filtered_data = data[(data['create_date_et'] >= start_date) & (data['create_date_et'] <= end_date)].sort_values(by=['create_date_et'])
filtered_data['create_date_et']

filtered_data.shape

## (10) Season and holiday

In [ ]:
months = closed_requests2['create_date_et'].dt.month
closed_requests['season'] = np.select(
    [months.isin([12,1,2]), months.isin([3,4,5]), months.isin([6,7,8])],
    [0, 1, 2],
    default=3
)

In [85]:
closed_requests.columns

Index(['_id', 'group_id', 'num_requests', 'parent_closed', 'status_name',
       'status_code', 'dept', 'request_type_name', 'request_type_id',
       'create_date_et', 'last_action_et', 'closed_date_et', 'origin', 'city',
       'neighborhood', 'council_district', 'ward', 'police_zone', 'latitude',
       'longitude', 'geo_accuracy', 'origin_Call Center',
       'origin_Control Panel', 'origin_Email', 'origin_QAlert Mobile iOS',
       'origin_Report2Gov Android', 'origin_Report2Gov Website',
       'origin_Report2Gov iOS', 'origin_Text Message', 'origin_Twitter',
       'origin_Website', 'request_type_id_corrected', 'time_taken',
       'request_type_grouped', 'season'],
      dtype='object')

In [90]:
closed_requests['season'].unique()

array([1, 2, 3, 0])

In [101]:
import holidays
us_holidays = holidays.US(years=closed_requests2['create_date_et'].dt.year.unique())
closed_requests2['is_holiday'] = closed_requests2['create_date_et'].apply(lambda x: x in us_holidays).astype(int)

In [103]:
closed_requests2['is_holiday']

151382    0
173037    0
196183    0
111096    0
413448    0
         ..
799995    0
799997    0
799994    0
799993    0
799999    0
Name: is_holiday, Length: 710557, dtype: int32

## (11) City

In [97]:
closed_requests['city'].unique()

array(['Pittsburgh', 'Munhall', 'Homestead', 'Carnegie', 'Mount Oliver',
       'Green Tree', 'Crafton', 'Wilkinsburg', 'CASTLE SHANN', 'ARSENAL',
       'Brentwood', 'Millvale', 'West Homestead', 'Swissvale', 'Baldwin',
       'Ingram', 'West Mifflin', 'Penn Hills', 'Avalon',
       'City of Pittsburgh', 'McKees Rocks', 'Dormont', 'Pitcairn',
       'Baldwin Township'], dtype=object)

In [115]:
closed_requests['city'].nunique()

24

In [95]:
city_counts = closed_requests['city'].value_counts()
city_counts

city
Pittsburgh            709549
Homestead                348
Carnegie                 294
Munhall                  139
Mount Oliver              66
Crafton                   49
Wilkinsburg               24
Green Tree                17
ARSENAL                   14
Dormont                   10
West Homestead             8
Millvale                   6
Swissvale                  6
Ingram                     5
Brentwood                  4
Baldwin                    3
Avalon                     3
West Mifflin               2
Penn Hills                 2
City of Pittsburgh         2
McKees Rocks               2
Baldwin Township           2
CASTLE SHANN               1
Pitcairn                   1
Name: count, dtype: int64

## (12) Dept

In [116]:
data['dept'].nunique()

62

In [76]:
data['dept'].unique()

array(['DOMI - TrafficShop', 'Permits, Licenses and Inspections',
       'DPW - Street Maintenance', 'DOMI - Permits', 'DOMI - Structures',
       'DOMI - Traffic', 'DPW - Refuse', '311', 'DPW - Forestry Division',
       'DPW - Park Maintenance', 'Police - AVU', 'DOMI - Streets',
       'DOMI - Asphalt', 'Pittsburgh Water and Sewer Authority',
       'Animal Care & Control', 'DOMI - Construction',
       'DPW - Administration', 'DPW - Construction Division',
       'Police - Zones 1-6', 'DPW - Facilities', 'DPW - 2nd Division',
       'Finance', 'Port Authority Transit', 'OMI', 'Fire Bureau', nan,
       'Parks & Recs-Programs', 'Parking Authority',
       'Allegheny City Electric',
       'City Planning - Bicycles/Pedestrian traffic', 'Lamar Advertising',
       'City Planning - Disabilities', 'Innovation & Performance',
       "Mayor's Office - Community Affairs", 'PCSC',
       'DOMI - TrafficPermits', 'School Guards', 'ACHD - Housing',
       'EMS - Administration',
       'City P

In [100]:
data.columns

Index(['_id', 'group_id', 'num_requests', 'parent_closed', 'status_name',
       'status_code', 'dept', 'request_type_name', 'request_type_id',
       'create_date_et', 'last_action_et', 'closed_date_et', 'origin', 'city',
       'neighborhood', 'council_district', 'ward', 'police_zone', 'latitude',
       'longitude', 'geo_accuracy', 'origin_Call Center',
       'origin_Control Panel', 'origin_Email', 'origin_QAlert Mobile iOS',
       'origin_Report2Gov Android', 'origin_Report2Gov Website',
       'origin_Report2Gov iOS', 'origin_Text Message', 'origin_Twitter',
       'origin_Website', 'request_type_id_corrected'],
      dtype='object')

# Sparse-Column-Identification

In [ ]:
from sklearn.feature_selection import VarianceThreshold

data_value = data.values

X = data_value[:, :-1]
y = data_value[:, -1]

print(X.shape, y.shape)

vt = VarianceThreshold()

X_sel = vt.fit_transform(X)
print(X_sel.shape)

# Train the model

In [112]:
final_features = [
    'num_requests', 'status_code', 'council_district', 'ward', 'police_zone','request_type_id',
    # 时间特征
    'season', 'is_holiday',
    
    # One-Hot特征
    'origin_Call Center','origin_Control Panel', 'origin_Email', 'origin_QAlert Mobile iOS',
    'origin_Report2Gov Android', 'origin_Report2Gov Website','origin_Report2Gov iOS', 'origin_Text Message', 'origin_Twitter',
     'origin_Website'
    
]

X = closed_requests[final_features]
y = closed_requests['time_taken']

## 1. XGBoost vs LightGBM vs CatBoost

### (1) Compare the base model

In [113]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, median_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

tss = TimeSeriesSplit(n_splits=5)
mae_scores, medae_scores = [], []

for train_idx, test_idx in tss.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # 初始化模型
    models = {
        'XGBoost': XGBRegressor(random_state=42, enable_categorical=True),
        'LightGBM': LGBMRegressor(random_state=42),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
    }
    
    fold_results = {}
    for name, model in models.items():
        # 训练模型
        model.fit(X_train, y_train)
        
        # 预测与评估
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        medae = median_absolute_error(y_test, y_pred)
        
        fold_results[name] = {'MAE': mae, 'MedAE': medae}
    
    # 记录每折结果
    mae_scores.append({k: v['MAE'] for k, v in fold_results.items()})
    medae_scores.append({k: v['MedAE'] for k, v in fold_results.items()})

# 计算平均性能
final_mae = pd.DataFrame(mae_scores).mean()
final_medae = pd.DataFrame(medae_scores).mean()

print("平均MAE:\n", final_mae)
print("\n平均MedAE:\n", final_medae)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001633 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 281
[LightGBM] [Info] Number of data points in the train set: 118427, number of used features: 14
[LightGBM] [Info] Start training from score 3757.809605
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007466 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 298
[LightGBM] [Info] Number of data points in the train set: 236853, number of used features: 16
[LightGBM] [Info] Start training from score 3257.452308
[LightGBM] [Warning] F

### (2) Cross validation

In [114]:
tss = TimeSeriesSplit(n_splits=5)
mae_scores, medae_scores = [], []

for train_idx, test_idx in tss.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # 初始化模型
    models = {
        'XGBoost': XGBRegressor(random_state=42, enable_categorical=True),
        'LightGBM': LGBMRegressor(random_state=42),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
    }
    
    fold_results = {}
    for name, model in models.items():
        # 训练模型
        model.fit(X_train, y_train)
        
        # 预测与评估
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        medae = median_absolute_error(y_test, y_pred)
        
        fold_results[name] = {'MAE': mae, 'MedAE': medae}
    
    # 记录每折结果
    mae_scores.append({k: v['MAE'] for k, v in fold_results.items()})
    medae_scores.append({k: v['MedAE'] for k, v in fold_results.items()})

# 计算平均性能
final_mae = pd.DataFrame(mae_scores).mean()
final_medae = pd.DataFrame(medae_scores).mean()

print("平均MAE:\n", final_mae)
print("\n平均MedAE:\n", final_medae)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002768 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 281
[LightGBM] [Info] Number of data points in the train set: 118427, number of used features: 14
[LightGBM] [Info] Start training from score 3757.809605
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 298
[LightGBM] [Info] Number of data points in the train set: 236853, number of used features: 16
[LightGBM] [Info] Start training from score 3257.452308
[LightGBM] [Warning] F